In [18]:
cd("/home/jzhang/gis_data/jzhang/MyAI")
using Pkg; 
import Pkg; 
Pkg.activate(".")

  Activating project at `~/MyAI`


In [2]:
Pkg.add("TextAnalysis")

   Resolving package versions...
  No Changes to `~/MyAI/Project.toml`
  No Changes to `~/MyAI/Manifest.toml`


In [3]:
using TextAnalysis

In [4]:
str = "To be or not to be ..."


"To be or not to be ..."

In [5]:
sd = StringDocument(str)

A StringDocument{String}
 * Language: Languages.English()
 * Title: Untitled Document
 * Author: Unknown Author
 * Timestamp: Unknown Time
 * Snippet: To be or not to be ...

In [6]:
td = TokenDocument("To be or not to be")

A TokenDocument{String}
 * Language: Languages.English()
 * Title: Untitled Document
 * Author: Unknown Author
 * Timestamp: Unknown Time
 * Snippet: ***SAMPLE TEXT NOT AVAILABLE***

In [7]:
ngd = NGramDocument("To be or not to be")

A NGramDocument{String}
 * Language: Languages.English()
 * Title: Untitled Document
 * Author: Unknown Author
 * Timestamp: Unknown Time
 * Snippet: ***SAMPLE TEXT NOT AVAILABLE***

In [8]:
tokens(sd)

6-element Vector{String}:
 "To"
 "be"
 "or"
 "not"
 "to"
 "be"

In [9]:
ngrams(sd)

Dict{String, Int64} with 5 entries:
  "or"  => 1
  "not" => 1
  "to"  => 1
  "To"  => 1
  "be"  => 2

In [32]:
#Pkg.add("GLM");
Pkg.add("CSV");
Pkg.add("PrettyPrinting");

   Resolving package versions...
  No Changes to `~/MyAI/Project.toml`
  No Changes to `~/MyAI/Manifest.toml`
   Resolving package versions...
  No Changes to `~/MyAI/Project.toml`
  No Changes to `~/MyAI/Manifest.toml`


In [37]:
using DataFrames;
using GLM;
using CSV;
using Gadfly;
using TextAnalysis;
using MLJ;
using Chain;
using Pipe;
using StableRNGs;

df = CSV.read("titanic-dataset.csv", DataFrame, header=true);
first(df, 10) |> pretty

┌─────────────┬──────────┬────────┬─────────────────────────────────────────────────────┬─────────┬────────────────────────────┬───────┬───────┬──────────────────┬────────────┬──────────────────────────┬─────────────────────────┐
│ PassengerId │ Survived │ Pclass │ Name                                                │ Sex     │ Age                        │ SibSp │ Parch │ Ticket           │ Fare       │ Cabin                    │ Embarked                │
│ Int64       │ Int64    │ Int64  │ String                                              │ String7 │ Union{Missing, Float64}    │ Int64 │ Int64 │ String31         │ Float64    │ Union{Missing, String15} │ Union{Missing, String1} │
│ Count       │ Count    │ Count  │ Textual                                             │ Textual │ Union{Missing, Continuous} │ Count │ Count │ Textual          │ Continuous │ Union{Missing, Textual}  │ Union{Missing, Textual} │
├─────────────┼──────────┼────────┼─────────────────────────────────────────────

In [38]:
delete!(df, :PassengerId);
delete!(df, :Name);
delete!(df, :Ticket);
delete!(df, :Cabin);
df[isna.(df[:Age]),:Age] = median(df[ .~isna.(df[:Age]),:Age])
df[isna.(df[:Embarked]),:Embarked] = "S"
pool!(df, [:Sex]);
pool!(df, [:Embarked]);

LoadError: MethodError: no method matching deleteat!(::DataFrame, ::Symbol)
The function `deleteat!` exists, but no method is defined for this combination of argument types.

[0mClosest candidates are:
[0m  deleteat!([91m::SentinelArrays.MissingVector[39m, ::Any)
[0m[90m   @[39m [33mSentinelArrays[39m [90m~/.julia/packages/SentinelArrays/vt8A6/src/[39m[90m[4mmissingvector.jl:57[24m[39m
[0m  deleteat!(::DataFrame, [91m::Colon[39m)
[0m[90m   @[39m [32mDataFrames[39m [90m~/.julia/packages/DataFrames/kcA9R/src/dataframe/[39m[90m[4mdataframe.jl:856[24m[39m
[0m  deleteat!(::DataFrame, [91m::Integer[39m)
[0m[90m   @[39m [32mDataFrames[39m [90m~/.julia/packages/DataFrames/kcA9R/src/dataframe/[39m[90m[4mdataframe.jl:858[24m[39m
[0m  ...


In [ ]:
split_pt = trunc(Int,size(df,1)*0.7) # 70% validation
shuffle_idx = sample(1:size(df,1),size(df,1));
df_train = df[1:split_pt,:];
df_validate = df[split_pt+1:size(df,1),:];

In [ ]:
model = glm(@formula(Survived ~ Pclass + Sex + Age + SibSp + Parch + Fare + Embarked), df_train, Binomial(), LogitLink());
pred = predict(model,df_validate);
pred = convert(DataArray{Int}, round.(pred));

In [ ]:
print("Accuracy: ")
println( sum(pred .== df_validate[:Survived]) / length(pred))